# E-commerce Recommendation System — Initial Models

## Objective

This notebook demonstrates the first two recommendation models:
1. **Content-Based Filtering** — Recommends items similar to a given item
2. **Collaborative Filtering** — Recommends items based on similar user behavior

We will evaluate both models using metrics like Precision@K, Recall@K, and NDCG@K.

---

## 1. Setup and Configuration

In [1]:
import logging
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Add project to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format="%(name)s - %(levelname)s - %(message)s",
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 50)

print(f"Project root: {project_root}")
print(f"Python version: {sys.version}")

Project root: c:\Users\ASUS\Documents\ecommerce-recommender
Python version: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]


## 2. Import Models and Utilities

In [2]:
from src.data.loader import (
    load_events,
    load_interactions,
    load_item_features,
    get_project_root,
)
from src.models.content_based import ContentBasedRecommender
from src.models.collaborative import CollaborativeFilteringRecommender
from src.evaluation.metrics import (
    prepare_train_test_split,
    prepare_ground_truth,
    evaluate_recommendations,
)

print("✓ Imports successful")

✓ Imports successful


## 3. Load Processed Data

In [3]:
# Load cleaned events
print("Loading events...")
events = load_events(processed=True)

print(f"\nEvents shape: {events.shape}")
print(f"Event types: {events['event'].unique()}")
print(f"Date range: {events['datetime'].min()} to {events['datetime'].max()}")
print(f"\nFirst few events:")
events.head()

src.data.loader - INFO - Loading events from c:\Users\ASUS\Documents\ecommerce-recommender\data\processed\events_clean.parquet


Loading events...


src.data.loader - INFO - Loaded 2755641 events



Events shape: (2755641, 13)
Event types: <ArrowStringArray>
['view', 'addtocart', 'transaction']
Length: 3, dtype: str
Date range: 2015-05-03 03:00:04.384000 to 2015-09-18 02:59:47.788000

First few events:


,timestamp,visitorid,event,itemid,transactionid,datetime,date,hour,day_of_week,day_name,month,interaction_weight,event_strength
0,1433221332117,257597,view,355908,NaN,2015-06-02 05:02:12.117,2015-06-02,5,1,Tuesday,6,1,1
1,1433224214164,992329,view,248676,NaN,2015-06-02 05:50:14.164,2015-06-02,5,1,Tuesday,6,1,1
2,1433221999827,111016,view,318965,NaN,2015-06-02 05:13:19.827,2015-06-02,5,1,Tuesday,6,1,1
3,1433221955914,483717,view,253185,NaN,2015-06-02 05:12:35.914,2015-06-02,5,1,Tuesday,6,1,1
4,1433221337106,951259,view,367447,NaN,2015-06-02 05:02:17.106,2015-06-02,5,1,Tuesday,6,1,1


In [4]:
# Load aggregated interactions
print("Loading interactions...")
interactions = load_interactions()

print(f"\nInteractions shape: {interactions.shape}")
print(f"Unique users: {interactions['visitorid'].nunique()}")
print(f"Unique items: {interactions['itemid'].nunique()}")
print(f"Sparsity: {1 - (len(interactions) / (interactions['visitorid'].nunique() * interactions['itemid'].nunique())):.4f}")
print(f"\nFirst few interactions:")
interactions.head()

src.data.loader - INFO - Loading interactions from c:\Users\ASUS\Documents\ecommerce-recommender\data\processed\interactions.parquet
src.data.loader - INFO - Loaded 2145179 user-item interactions


Loading interactions...

Interactions shape: (2145179, 6)
Unique users: 1407580
Unique items: 235061
Sparsity: 1.0000

First few interactions:


,visitorid,itemid,interaction_score,interaction_count,strongest_event,last_interaction
0,0,67045,1,1,view,2015-09-11 20:55:17.175
1,0,285930,1,1,view,2015-09-11 20:49:49.439
2,0,357564,1,1,view,2015-09-11 20:52:39.591
3,1,72028,1,1,view,2015-08-13 17:46:06.444
4,2,216305,2,2,view,2015-08-07 18:17:43.170


In [5]:
# Load item features
print("Loading item features...")
try:
    item_features = load_item_features()
    print(f"✓ Item features loaded successfully")
    print(f"\nItem features shape: {item_features.shape}")
    print(f"Columns: {item_features.columns.tolist()}")
    print(f"\nFirst few items:")
    print(item_features.head())
except FileNotFoundError as e:
    print(f"⚠ Item features not found: {e}")
    print("Please run: python generate_item_features.py")

src.data.loader - INFO - Loading item features from c:\Users\ASUS\Documents\ecommerce-recommender\data\processed\item_features.parquet
src.data.loader - INFO - Loaded features for 417053 items


Loading item features...
✓ Item features loaded successfully

Item features shape: (417053, 5)
Columns: ['itemid', 'categoryid', 'category_in_top', 'has_parent', 'num_properties']

First few items:
   itemid  categoryid  category_in_top  has_parent  num_properties
0       0         209             True           1              28
1       1        1114             True           1              35
2       2        1305             True           1              24
3       3        1171             True           1              29
4       4        1038             True           1              25


## 4. Prepare Train/Test Split

In [6]:
# Time-aware train/test split (80/20)
print("Preparing time-aware train/test split...")
train_interactions, test_interactions = prepare_train_test_split(
    interactions,
    test_ratio=0.2,
    random_seed=42,
    time_aware=True,
    min_interactions=5,
)

print(f"\nTrain set: {len(train_interactions)} interactions")
print(f"Test set: {len(test_interactions)} interactions")

# Prepare ground truth from test set
test_ground_truth = prepare_ground_truth(test_interactions)
print(f"\nTest ground truth: {len(test_ground_truth)} users with test items")

src.evaluation.metrics - INFO - Preparing train/test split


Preparing time-aware train/test split...


src.evaluation.metrics - INFO - Users/interactions after minimum-history filtering: 427236 interactions
src.evaluation.metrics - INFO - Train: 351913 interactions, Test: 75323 interactions



Train set: 351913 interactions
Test set: 75323 interactions

Test ground truth: 39635 users with test items


## 5. Model 1: Content-Based Filtering

In [7]:
print("Training Content-Based Recommender...")
cb_recommender = ContentBasedRecommender()
cb_recommender.fit(item_features)

model_info = cb_recommender.get_model_info()
print(f"\n✓ Content-Based Model Fitted")
for key, value in model_info.items():
    print(f"  {key}: {value}")

src.models.content_based - INFO - Fitting content-based model
src.models.content_based - INFO - Mapping 417053 items
src.features.item_features - INFO - Creating sparse item feature matrix


Training Content-Based Recommender...


src.features.item_features - INFO - Number of items: 417053
src.features.item_features - INFO - Unique property/value features: 2416027; retained after frequency filtering: 255585 (min frequency=5)
src.features.item_features - INFO - Loading category hierarchy
src.data.loader - INFO - Loading category tree from c:\Users\ASUS\Documents\ecommerce-recommender\data\raw\category_tree.csv
src.data.loader - INFO - Loaded 1669 categories
src.features.item_features - INFO - Loaded 1669 categories
src.features.item_features - INFO - Feature matrix shape: (417053, 257133) (items x features)
src.features.item_features - INFO - Feature matrix non-zero values: 11375238
src.features.item_features - INFO - Feature matrix sparsity: 99.99%
src.models.content_based - INFO - Building KNN model for similarity search
src.models.content_based - INFO - Model fitted with 417053 items



✓ Content-Based Model Fitted
  status: fitted
  n_items: 417053
  feature_matrix_shape: (417053, 257133)
  feature_matrix_sparsity: 0.9998939253944109
  n_property_value_features: 2416027
  n_property_value_features_retained: 255585
  n_neighbors_used: 100


In [8]:
# Example recommendations using Content-Based model
print("Content-Based Filtering — Example Recommendations")
print("=" * 60)

# Select some popular items for demonstration
sample_items = interactions['itemid'].value_counts().head(5).index.tolist()

for item_id in sample_items:
    recommendations = cb_recommender.recommend(item_id, k=5)
    print(f"\nQuery item: {item_id}")
    print(f"Similar items:")
    for rec_item, score in recommendations:
        print(f"  - Item {rec_item}: similarity = {score:.4f}")

Content-Based Filtering — Example Recommendations

Query item: 187946
Similar items:
  - Item 240139: similarity = 0.6262
  - Item 87319: similarity = 0.6262
  - Item 245247: similarity = 0.6262
  - Item 298220: similarity = 0.6262
  - Item 80667: similarity = 0.6262

Query item: 5411
Similar items:
  - Item 395853: similarity = 0.7650
  - Item 74218: similarity = 0.7414
  - Item 408923: similarity = 0.7370
  - Item 219766: similarity = 0.7241
  - Item 133751: similarity = 0.7241

Query item: 370653
Similar items:
  - Item 250539: similarity = 0.7700
  - Item 159256: similarity = 0.7437
  - Item 256853: similarity = 0.7437
  - Item 445237: similarity = 0.7315
  - Item 335523: similarity = 0.7315

Query item: 461686
Similar items:
  - Item 155737: similarity = 0.7686
  - Item 307322: similarity = 0.7144
  - Item 312929: similarity = 0.6763
  - Item 339032: similarity = 0.6663
  - Item 445258: similarity = 0.6640

Query item: 219512
Similar items:
  - Item 1447: similarity = 0.7721
  - I

In [15]:
# Evaluate Content-Based model on a sample of test users
print("Evaluating Content-Based Model...")

# Use the same users for both models
sample_users = list(test_ground_truth.keys())[:100]

sample_test_ground_truth = {
    user_id: test_ground_truth[user_id]
    for user_id in sample_users
}

cb_predictions = {}

train_user_items = prepare_ground_truth(train_interactions)

for user_id in sample_users:

    if user_id not in train_user_items:
        continue

    user_train_items = train_user_items[user_id]

    # Only use items known by the Content-Based model
    known_items = [
        item
        for item in user_train_items
        if item in cb_recommender.item_id_to_idx
    ]

    if not known_items:
        continue

    user_recommendations = []

    # Use at most 3 items from the user's history
    for train_item in known_items[:3]:

        recs = cb_recommender.recommend(
            train_item,
            k=10,
            exclude_items=user_train_items,
        )

        for rec_item, score in recs:
            user_recommendations.append(rec_item)

    # Remove duplicates and keep top 10
    cb_predictions[user_id] = list(
        dict.fromkeys(user_recommendations)
    )[:10]

print(f"✓ Generated predictions for {len(cb_predictions)} users")


# Evaluate
cb_results = evaluate_recommendations(
    cb_predictions,
    sample_test_ground_truth,
    k_values=[5, 10],
)

print("\n" + "=" * 60)
print("Content-Based Filtering Evaluation Results")
print("=" * 60)

for metric, value in cb_results.items():
    if not metric.endswith("_std") and metric != "evaluated_users":
        print(
            f"{metric}: {value:.4f} "
            f"(±{cb_results[f'{metric}_std']:.4f})"
        )

print(f"Evaluated users: {cb_results['evaluated_users']}")

Evaluating Content-Based Model...


src.evaluation.metrics - INFO - Evaluating recommendations for 100 users
src.evaluation.metrics - INFO - Evaluated 100 users
src.evaluation.metrics - INFO -   Precision@5: 0.0160 (±0.0543)
src.evaluation.metrics - INFO -   Recall@5: 0.0650 (±0.2308)
src.evaluation.metrics - INFO -   NDCG@5: 0.0372 (±0.1397)
src.evaluation.metrics - INFO -   Precision@10: 0.0120 (±0.0325)
src.evaluation.metrics - INFO -   Recall@10: 0.1000 (±0.2828)
src.evaluation.metrics - INFO -   NDCG@10: 0.0492 (±0.1491)


✓ Generated predictions for 100 users

Content-Based Filtering Evaluation Results
precision@5: 0.0160 (±0.0543)
precision@10: 0.0120 (±0.0325)
recall@5: 0.0650 (±0.2308)
recall@10: 0.1000 (±0.2828)
ndcg@5: 0.0372 (±0.1397)
ndcg@10: 0.0492 (±0.1491)
Evaluated users: 100


## 6. Model 2: Collaborative Filtering

In [11]:
print("Training Collaborative Filtering Recommender...")
cf_recommender = CollaborativeFilteringRecommender(min_interactions=2)
cf_recommender.fit(train_interactions)

model_info = cf_recommender.get_model_info()
print(f"\n✓ Collaborative Filtering Model Fitted")
for key, value in model_info.items():
    print(f"  {key}: {value}")

src.models.collaborative - INFO - Fitting collaborative filtering model
src.models.collaborative - INFO - Building interaction matrix
src.models.collaborative - INFO - Filtered to 39635 users with >= 2 interactions
src.models.collaborative - INFO - User IDs: 39635, Item IDs: 83876


Training Collaborative Filtering Recommender...


src.models.collaborative - INFO - Interaction matrix shape: (39635, 83876), Sparsity: 99.99%
src.models.collaborative - INFO - Normalizing interaction matrix
src.models.collaborative - INFO - Computing user similarity matrix
src.models.collaborative - INFO - Collaborative filtering model fitted



✓ Collaborative Filtering Model Fitted
  status: fitted
  n_users: 39635
  n_items: 83876
  matrix_shape: (39635, 83876)
  sparsity: 0.9998941432059748
  min_interactions: 2


In [12]:
# Example recommendations using Collaborative Filtering
print("Collaborative Filtering — Example Recommendations")
print("=" * 60)

# Select some active users for demonstration
sample_users = interactions['visitorid'].value_counts().head(5).index.tolist()

for user_id in sample_users:
    recommendations = cf_recommender.recommend(user_id, k=5)
    print(f"\nUser: {user_id}")
    print(f"Recommended items:")
    for rec_item, score in recommendations:
        print(f"  - Item {rec_item}: score = {score:.4f}")

Collaborative Filtering — Example Recommendations

User: 1150086
Recommended items:
  - Item 48030: score = 5.4313
  - Item 213834: score = 4.7944
  - Item 409804: score = 4.3640
  - Item 445351: score = 4.2934
  - Item 174724: score = 4.2177

User: 530559
Recommended items:
  - Item 213834: score = 10.8021
  - Item 409804: score = 7.8077
  - Item 355994: score = 4.7635
  - Item 445351: score = 3.2047
  - Item 37029: score = 2.1880

User: 892013
Recommended items:
  - Item 186820: score = 3.8797
  - Item 183226: score = 2.4703
  - Item 104651: score = 2.1002
  - Item 284176: score = 1.6187
  - Item 29196: score = 1.5387

User: 895999
Recommended items:
  - Item 12217: score = 10.5178
  - Item 294676: score = 4.7647
  - Item 257655: score = 4.5539
  - Item 46156: score = 3.9092
  - Item 166306: score = 3.8870

User: 152963
Recommended items:
  - Item 9877: score = 15.3730
  - Item 420960: score = 4.0769
  - Item 325310: score = 3.2131
  - Item 248455: score = 2.9561
  - Item 283115: sco

In [13]:
# Evaluate Collaborative Filtering model on test set
print("Evaluating Collaborative Filtering Model...")

# Generate recommendations for test users
cf_predictions = cf_recommender.recommend_batch(
    list(test_ground_truth.keys()),
    k=10,
)

# Convert (item_id, score) tuples to just item_ids
cf_predictions = {user_id: [item for item, _ in recs] for user_id, recs in cf_predictions.items()}

print(f"✓ Generated predictions for {len(cf_predictions)} users")

# Evaluate
cf_results = evaluate_recommendations(
    cf_predictions,
    test_ground_truth,
    k_values=[5, 10],
)

print("\n" + "=" * 60)
print("Collaborative Filtering Evaluation Results")
print("=" * 60)
for metric, value in cf_results.items():
    if not metric.endswith("_std") and metric != "evaluated_users":
        print(f"{metric}: {value:.4f} (±{cf_results[f'{metric}_std']:.4f})")
print(f"Evaluated users: {cf_results['evaluated_users']}")

Evaluating Collaborative Filtering Model...


src.evaluation.metrics - INFO - Evaluating recommendations for 39635 users


✓ Generated predictions for 39635 users


src.evaluation.metrics - INFO - Evaluated 39635 users
src.evaluation.metrics - INFO -   Precision@5: 0.0127 (±0.0500)
src.evaluation.metrics - INFO -   Recall@5: 0.0529 (±0.2167)
src.evaluation.metrics - INFO -   NDCG@5: 0.0271 (±0.1122)
src.evaluation.metrics - INFO -   Precision@10: 0.0136 (±0.0363)
src.evaluation.metrics - INFO -   Recall@10: 0.1108 (±0.3026)
src.evaluation.metrics - INFO -   NDCG@10: 0.0466 (±0.1312)



Collaborative Filtering Evaluation Results
precision@5: 0.0127 (±0.0500)
precision@10: 0.0136 (±0.0363)
recall@5: 0.0529 (±0.2167)
recall@10: 0.1108 (±0.3026)
ndcg@5: 0.0271 (±0.1122)
ndcg@10: 0.0466 (±0.1312)
Evaluated users: 39635


## 7. Model Comparison

In [16]:
# Compare results
import pandas as pd

comparison_data = []

for k in [5, 10]:
    comparison_data.append({
        "Model": "Content-Based",
        "Metric": f"Precision@{k}",
        "Value": f"{cb_results[f'precision@{k}']:.4f}",
        "Std": f"{cb_results[f'precision@{k}_std']:.4f}",
    })
    comparison_data.append({
        "Model": "Content-Based",
        "Metric": f"Recall@{k}",
        "Value": f"{cb_results[f'recall@{k}']:.4f}",
        "Std": f"{cb_results[f'recall@{k}_std']:.4f}",
    })
    comparison_data.append({
        "Model": "Content-Based",
        "Metric": f"NDCG@{k}",
        "Value": f"{cb_results[f'ndcg@{k}']:.4f}",
        "Std": f"{cb_results[f'ndcg@{k}_std']:.4f}",
    })
    comparison_data.append({
        "Model": "Collaborative Filtering",
        "Metric": f"Precision@{k}",
        "Value": f"{cf_results[f'precision@{k}']:.4f}",
        "Std": f"{cf_results[f'precision@{k}_std']:.4f}",
    })
    comparison_data.append({
        "Model": "Collaborative Filtering",
        "Metric": f"Recall@{k}",
        "Value": f"{cf_results[f'recall@{k}']:.4f}",
        "Std": f"{cf_results[f'recall@{k}_std']:.4f}",
    })
    comparison_data.append({
        "Model": "Collaborative Filtering",
        "Metric": f"NDCG@{k}",
        "Value": f"{cf_results[f'ndcg@{k}']:.4f}",
        "Std": f"{cf_results[f'ndcg@{k}_std']:.4f}",
    })

comparison_df = pd.DataFrame(comparison_data)
print("\nComparison of Content-Based and Collaborative Filtering Models")
print("=" * 80)
print(comparison_df.to_string(index=False))


Comparison of Content-Based and Collaborative Filtering Models
                  Model       Metric  Value    Std
          Content-Based  Precision@5 0.0160 0.0543
          Content-Based     Recall@5 0.0650 0.2308
          Content-Based       NDCG@5 0.0372 0.1397
Collaborative Filtering  Precision@5 0.0127 0.0500
Collaborative Filtering     Recall@5 0.0529 0.2167
Collaborative Filtering       NDCG@5 0.0271 0.1122
          Content-Based Precision@10 0.0120 0.0325
          Content-Based    Recall@10 0.1000 0.2828
          Content-Based      NDCG@10 0.0492 0.1491
Collaborative Filtering Precision@10 0.0136 0.0363
Collaborative Filtering    Recall@10 0.1108 0.3026
Collaborative Filtering      NDCG@10 0.0466 0.1312


## 8. Model Limitations and Insights

In [17]:
print("""
╔════════════════════════════════════════════════════════════════════════════╗
║                          MODEL LIMITATIONS & INSIGHTS                       ║
╚════════════════════════════════════════════════════════════════════════════╝

📊 CONTENT-BASED FILTERING
───────────────────────────
Strengths:
  ✓ Captures product similarity based on features (category, properties)
  ✓ No cold-start problem for new items with features
  ✓ Transparent: can explain why items were recommended
  ✓ Handles new items well if they have feature vectors

Limitations:
  ✗ Requires meaningful product features (challenge: properties are anonymized)
  ✗ Cannot discover serendipitous or unexpected items
  ✗ Homogeneous recommendations (recommends similar items only)
  ✗ Cannot leverage user behavior patterns or collective intelligence
  ✗ Items with similar categories will always be similar

Recommendation Quality:
  - Depends heavily on quality of item features
  - Current features (category + properties) are basic
  - Would benefit from more rich item representations (e.g., embeddings)


👥 COLLABORATIVE FILTERING
──────────────────────────
Strengths:
  ✓ Leverages collective user behavior patterns
  ✓ Can discover serendipitous recommendations
  ✓ Captures complex preference patterns implicitly
  ✓ No need for explicit item features
  ✓ Works well for users with many interactions

Limitations:
  ✗ Cold-start problem for new users with no interactions
  ✗ Cold-start problem for new items with no interactions
  ✗ Suffers from sparsity (most user-item pairs have no interaction)
  ✗ Popular items tend to be over-recommended
  ✗ Cannot explain recommendations easily to users
  ✗ Requires sufficient training data (dense interactions)

Recommendation Quality:
  - Varies significantly by user popularity
  - Active users with many interactions: better recommendations
  - Inactive users: falls back to popular items
  - Current sparsity is a challenge


🔄 COMPARISON SUMMARY
─────────────────────
1. Content-Based scores: {:.4f} - {:.4f} (Precision@10)
2. Collaborative scores: {:.4f} - {:.4f} (Precision@10)

The differences in performance depend on:
  • Data sparsity and user activity distribution
  • Feature quality for content-based (current: categories only)
  • User-item interaction density for collaborative filtering
  • Evaluation methodology and test set composition


🎯 NEXT STEPS (Milestone 2)
───────────────────────────
1. Implement HYBRID model combining both approaches
2. Enhance item features with:
   - Product embeddings
   - Learned feature representations
   - More sophisticated feature engineering
3. Optimize collaborative filtering:
   - Matrix factorization techniques
   - Deep learning approaches
4. Implement:
   - Advanced evaluation techniques
   - A/B testing framework
   - FastAPI backend for serving predictions
   - React frontend for recommendations UI
5. Deploy the system
""".format(
        cb_results['precision@10'],
        cb_results['precision@5'],
        cf_results['precision@10'],
        cf_results['precision@5']
    )
)


╔════════════════════════════════════════════════════════════════════════════╗
║                          MODEL LIMITATIONS & INSIGHTS                       ║
╚════════════════════════════════════════════════════════════════════════════╝

📊 CONTENT-BASED FILTERING
───────────────────────────
Strengths:
  ✓ Captures product similarity based on features (category, properties)
  ✓ No cold-start problem for new items with features
  ✓ Transparent: can explain why items were recommended
  ✓ Handles new items well if they have feature vectors

Limitations:
  ✗ Requires meaningful product features (challenge: properties are anonymized)
  ✗ Cannot discover serendipitous or unexpected items
  ✗ Homogeneous recommendations (recommends similar items only)
  ✗ Cannot leverage user behavior patterns or collective intelligence
  ✗ Items with similar categories will always be similar

Recommendation Quality:
  - Depends heavily on quality of item features
  - Current features (category + properties)

## 9. Summary

In [18]:
print("""
╔════════════════════════════════════════════════════════════════════════════╗
║                       STAGE 1 - DELIVERABLES SUMMARY                        ║
╚════════════════════════════════════════════════════════════════════════════╝

✅ COMPLETED:
  1. Data Understanding & Preparation (Stage 1)
     • EDA notebook with dataset analysis
     • Cleaned events data (2.7M records)
     • Aggregated user-item interactions (2.1M interactions)
     • Extracted item features from properties and categories

  2. Recommendation Models Implemented
     • Content-Based Filtering ✓
       - Builds item similarity from features
       - Recommends similar products
     
     • Collaborative Filtering ✓
       - Analyzes user behavior patterns
       - Finds similar users and recommends their items

  3. Evaluation Framework
     • Precision@K, Recall@K, NDCG@K metrics ✓
     • Time-aware train/test split ✓
     • Batch evaluation across multiple users ✓

  4. Code Organization
     • Modular architecture with reusable components
     • Type hints and comprehensive docstrings
     • Logging throughout the pipeline
     • No hardcoded paths

📊 PROJECT PROGRESS: ~40% Complete
   • Stage 1 (Data Preparation): ~100% ✓
   • Stage 2 (Models): ~40% (2 of 5 planned models) ✓
   • Stage 3 (Application): 0% (planned for next milestone)

📁 FILES CREATED:
   • src/data/loader.py — Data loading and management
   • src/features/item_features.py — Item feature engineering
   • src/models/content_based.py — Content-based recommender
   • src/models/collaborative.py — Collaborative filtering
   • src/evaluation/metrics.py — Evaluation metrics
   • notebooks/02_initial_models.ipynb — This notebook
   • generate_item_features.py — Feature generation script

🚀 READY FOR NEXT MILESTONE:
   • Hybrid model implementation
   • Advanced feature engineering
   • FastAPI backend development
   • React frontend development
   • Deployment setup

""")


╔════════════════════════════════════════════════════════════════════════════╗
║                       STAGE 1 - DELIVERABLES SUMMARY                        ║
╚════════════════════════════════════════════════════════════════════════════╝

✅ COMPLETED:
  1. Data Understanding & Preparation (Stage 1)
     • EDA notebook with dataset analysis
     • Cleaned events data (2.7M records)
     • Aggregated user-item interactions (2.1M interactions)
     • Extracted item features from properties and categories

  2. Recommendation Models Implemented
     • Content-Based Filtering ✓
       - Builds item similarity from features
       - Recommends similar products

     • Collaborative Filtering ✓
       - Analyzes user behavior patterns
       - Finds similar users and recommends their items

  3. Evaluation Framework
     • Precision@K, Recall@K, NDCG@K metrics ✓
     • Time-aware train/test split ✓
     • Batch evaluation across multiple users ✓

  4. Code Organization
     • Modular archite